# Phase 3: Statistical Hypothesis Testing

"Treatment looks better in every metric. But could this have happened just by random chance?"

H0 (Null Hypothesis):     There is NO real difference between control and treatment.
                          Any gap we see is just random chance.

H1 (Alternative):         There IS a real difference. Treatment genuinely performs better.

Which Test for Which Metric 
CTR (clicked: yes/no)Chi-Square testComparing two proportions
Dwell Time (seconds)T-testComparing two averages
Bounce Rate (yes/no)Chi-Square testComparing two proportions

## Setup:

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('search_experiment.csv')

control   = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

print(f"Control sessions:   {len(control)}")
print(f"Treatment sessions: {len(treatment)}")

Control sessions:   10082
Treatment sessions: 9931


## Chi-Square test on CTR:

In [2]:
# Build a contingency table
#              clicked    not clicked
# control        x             x
# treatment      x             x

control_clicked   = control['clicked'].sum()
control_not       = len(control) - control_clicked
treatment_clicked = treatment['clicked'].sum()
treatment_not     = len(treatment) - treatment_clicked

contingency_table = [
    [control_clicked,   control_not],
    [treatment_clicked, treatment_not]
]

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print("=== CTR: Chi-Square Test ===")
print(f"Chi2 statistic : {chi2:.4f}")
print(f"P-value        : {p_value:.6f}")
print(f"Degrees of freedom: {dof}")
print()
if p_value < 0.05:
    print("RESULT: Statistically significant. We REJECT the null hypothesis.")
    print("The CTR difference is NOT due to random chance.")
else:
    print("RESULT: NOT significant. We FAIL to reject the null hypothesis.")

=== CTR: Chi-Square Test ===
Chi2 statistic : 81.9642
P-value        : 0.000000
Degrees of freedom: 1

RESULT: Statistically significant. We REJECT the null hypothesis.
The CTR difference is NOT due to random chance.


## T-test on Dwell Time:

In [3]:
# Only use sessions where user actually clicked
control_dwell   = control[control['clicked'] == 1]['dwell_time_sec']
treatment_dwell = treatment[treatment['clicked'] == 1]['dwell_time_sec']

t_stat, p_value_dwell = stats.ttest_ind(control_dwell, treatment_dwell)

print("=== Dwell Time: Independent T-Test ===")
print(f"Control mean   : {control_dwell.mean():.2f} sec")
print(f"Treatment mean : {treatment_dwell.mean():.2f} sec")
print(f"T-statistic    : {t_stat:.4f}")
print(f"P-value        : {p_value_dwell:.6f}")
print()
if p_value_dwell < 0.05:
    print("RESULT: Statistically significant. We REJECT the null hypothesis.")
    print("The dwell time difference is NOT due to random chance.")
else:
    print("RESULT: NOT significant. We FAIL to reject the null hypothesis.")

=== Dwell Time: Independent T-Test ===
Control mean   : 151.29 sec
Treatment mean : 180.71 sec
T-statistic    : -24.8710
P-value        : 0.000000

RESULT: Statistically significant. We REJECT the null hypothesis.
The dwell time difference is NOT due to random chance.


##  Chi-Square test on Bounce Rate:

In [4]:
control_bounced     = control['bounced'].sum()
control_not_bounced = len(control) - control_bounced
treatment_bounced     = treatment['bounced'].sum()
treatment_not_bounced = len(treatment) - treatment_bounced

bounce_table = [
    [control_bounced,   control_not_bounced],
    [treatment_bounced, treatment_not_bounced]
]

chi2_bounce, p_bounce, dof_bounce, _ = stats.chi2_contingency(bounce_table)

print("=== Bounce Rate: Chi-Square Test ===")
print(f"Chi2 statistic : {chi2_bounce:.4f}")
print(f"P-value        : {p_bounce:.6f}")
print()
if p_bounce < 0.05:
    print("RESULT: Statistically significant. We REJECT the null hypothesis.")
    print("The bounce rate difference is NOT due to random chance.")
else:
    print("RESULT: NOT significant. We FAIL to reject the null hypothesis.")

=== Bounce Rate: Chi-Square Test ===
Chi2 statistic : 223.8899
P-value        : 0.000000

RESULT: Statistically significant. We REJECT the null hypothesis.
The bounce rate difference is NOT due to random chance.


## Effect Size (Cohen's d for dwell time):

In [5]:
# p-value tells us IF the difference is real
# Cohen's d tells us HOW BIG the difference is
# This is what separates junior from senior analysts

mean_diff = treatment_dwell.mean() - control_dwell.mean()
pooled_std = np.sqrt(
    (control_dwell.std()**2 + treatment_dwell.std()**2) / 2
)
cohens_d = mean_diff / pooled_std

print("=== Effect Size: Cohen's d (Dwell Time) ===")
print(f"Cohen's d: {cohens_d:.4f}")
print()
if abs(cohens_d) < 0.2:
    print("Effect size: SMALL")
elif abs(cohens_d) < 0.5:
    print("Effect size: MEDIUM")
else:
    print("Effect size: LARGE")

=== Effect Size: Cohen's d (Dwell Time) ===
Cohen's d: 0.4899

Effect size: MEDIUM


Every test rejected the null hypothesis. The improvements are real — not luck.